In [ ]:
pip install -U langchain langchain-core langchain-community langchain-text-splitters langchain-nvidia-ai-endpoints faiss-cpu python-dotenv

In [ ]:
# 🔷 Question 1: Build an End-to-End RAG + Agent System (25 Marks)
# 🧩 Scenario
# You are an AI intern at an ed-tech company. Students frequently ask questions about:

# Course policies (refunds, deadlines)
# Lecture content (PDF notes)
# Assignment deadlines
# Their enrollment status
# The company wants a single intelligent assistant that:

# Answers questions using internal documents (PDFs, FAQs)
# Fetches student-specific data (like enrollment or progress) using tools/APIs
# Avoids hallucination and gives reliable answers
# 💻 Task
# Design and implement a working prototype (pseudo-code or real code) for this system.

# Your solution must include:

# ✅ 1. RAG Pipeline
# How you will ingest and preprocess documents
# Chunking strategy (with justification)
# Embedding + vector store choice
# Retrieval logic
# How context is injected into the LLM
# ✅ 2. Agent Integration
# Design an agent that decides:
# When to use RAG
# When to call a tool (e.g., get_student_status(student_id))
# Show how tools are defined and used
# ✅ 3. End-to-End Flow
# Write code or structured pseudo-code showing:

# Input query
# Retrieval step
# Tool calling (if needed)
# Final answer generation
# ✅ 4. Reliability Improvements
# Show at least 2 techniques in code/design to:

# Reduce hallucination
# Improve answer quality
# 🎯 Expected Outcome
# A working architecture/code that demonstrates:

# RAG + Agent working together
# Decision-making capability
# Real-world practicality


In [3]:
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import Tool

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_community.embeddings import HuggingFaceEmbeddings


# =====================
# LOAD ENV
# =====================
load_dotenv()

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")


# =====================
# NVIDIA LLM
# =====================
llm = ChatNVIDIA(
    model="meta/llama3-8b-instruct",
    api_key=NVIDIA_API_KEY,
    temperature=0
)


# =====================
# HARDCODE DOCUMENTS
# =====================
def load_documents():

    docs = [

        Document(
            page_content="""
            Refund Policy:
            Students can request refund within 7 days of enrollment.
            After 7 days refund is not allowed.
            Refund processed in 5 working days.
            Refund is not available once certificate is issued.
            """,
            metadata={"source": "policy_refund"}
        ),

        Document(
            page_content="""
            Enrollment Policy:
            Students must complete enrollment by submitting valid details.
            Enrollment confirmation is sent via email.
            Students can enroll in multiple courses.
            """,
            metadata={"source": "policy_enrollment"}
        ),

        Document(
            page_content="""
            Assignment Policy:
            Assignment must be submitted before deadline.
            Late submission allowed within 2 days with 10 percent penalty.
            No submission accepted after 2 days of deadline.
            Assignments are graded within 5 working days.
            """,
            metadata={"source": "policy_assignment"}
        ),

        Document(
            page_content="""
            Attendance Policy:
            Students must attend at least 70 percent lectures.
            Low attendance may affect certification eligibility.
            Attendance is automatically tracked.
            """,
            metadata={"source": "policy_attendance"}
        ),

        Document(
            page_content="""
            Lecture Content:
            Machine Learning lecture covers supervised learning,
            unsupervised learning, regression, classification,
            clustering and model evaluation techniques.
            """,
            metadata={"source": "lecture_ml"}
        ),

        Document(
            page_content="""
            Lecture Content:
            Deep Learning lecture includes neural networks,
            activation functions, backpropagation,
            convolution neural networks and recurrent neural networks.
            """,
            metadata={"source": "lecture_dl"}
        ),

        Document(
            page_content="""
            Lecture Content:
            Python programming basics include variables,
            loops, functions, lists, dictionaries and file handling.
            """,
            metadata={"source": "lecture_python"}
        ),

        Document(
            page_content="""
            Course FAQ:
            Course duration is 3 months.
            Students can access lectures anytime.
            Certificate provided after completion.
            Course includes quizzes and assignments.
            """,
            metadata={"source": "faq_course"}
        ),

        Document(
            page_content="""
            Certificate Policy:
            Certificate is provided after completing all modules.
            Minimum 50 percent marks required in assignments.
            Certificate is downloadable from dashboard.
            """,
            metadata={"source": "faq_certificate"}
        ),

        Document(
            page_content="""
            Payment FAQ:
            Payment can be made using credit card,
            debit card, UPI and net banking.
            EMI option available for selected courses.
            """,
            metadata={"source": "faq_payment"}
        ),

        Document(
            page_content="""
            Support Information:
            Students can contact support via email or chat.
            Support response time is 24 hours.
            Technical issues are resolved within 48 hours.
            """,
            metadata={"source": "support"}
        ),

        Document(
            page_content="""
            Project Guidelines:
            Final project must be submitted before course completion.
            Project should demonstrate practical implementation.
            Plagiarism is strictly prohibited.
            """,
            metadata={"source": "project"}
        ),

        Document(
            page_content="""
            Quiz Policy:
            Weekly quizzes are conducted.
            Quiz attempts are limited to 3.
            Best score is considered for evaluation.
            """,
            metadata={"source": "quiz"}
        )

    ]

    return docs

# =====================
# CHUNKING
# =====================
def chunk_documents(docs):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=50
    )

    return splitter.split_documents(docs)


# =====================
# VECTOR DB
# =====================
def create_vector_db(chunks):

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vector_db = FAISS.from_documents(
        chunks,
        embeddings
    )

    return vector_db


# =====================
# RAG SEARCH
# =====================
def rag_search(query, vector_db):

    retriever = vector_db.as_retriever(
        search_kwargs={"k": 3}
    )

    docs = retriever.invoke(query)

    context = "\n".join(
        doc.page_content for doc in docs
    )

    return context, docs


# =====================
# TOOL
# =====================
def get_student_status(student_id):

    database = {

        "101": {
            "name": "Rahul",
            "course": "AI Fundamentals",
            "progress": "65%",
            "deadline": "25 March 2026"
        },

        "102": {
            "name": "Priya",
            "course": "Machine Learning",
            "progress": "80%",
            "deadline": "28 March 2026"
        }

    }

    return database.get(student_id, "Student not found")


student_tool = Tool(
    name="student_info",
    func=get_student_status,
    description="Get student progress and deadline using id"
)


# =====================
# ID EXTRACT
# =====================
def extract_id(query):

    for w in query.split():
        if w.isdigit():
            return w

    return None


# =====================
# AGENT ROUTER
# =====================
def agent_router(query):

    prompt = f"""

    Decide:

    1 → RAG (policies, lecture, faq)

    2 → TOOL (student progress, deadline)

    Query:
    {query}

    output only 1 or 2
    """

    return llm.invoke(prompt).content.strip()


# =====================
# RAG ANSWER
# =====================
def generate_rag_answer(query, context, docs):

    sources = [d.metadata["source"] for d in docs]

    prompt = f"""

    Answer ONLY from context.

    If not found say:
    I don't know.

    Context:
    {context}

    Question:
    {query}
    """

    ans = llm.invoke(prompt).content

    return f"""
{ans}

Sources:
{sources}
"""


# =====================
# TOOL ANSWER
# =====================
def generate_tool_answer(data):

    prompt = f"""
    Explain clearly:

    {data}
    """

    return llm.invoke(prompt).content


# =====================
# BUILD
# =====================
def build_system():

    docs = load_documents()

    chunks = chunk_documents(docs)

    return create_vector_db(chunks)


# =====================
# MAIN
# =====================
def edtech_assistant(query, vector_db):

    decision = agent_router(query)

    if decision == "1":

        context, docs = rag_search(query, vector_db)

        return generate_rag_answer(query, context, docs)


    elif decision == "2":

        sid = extract_id(query)

        if not sid:
            return "provide student id"

        data = get_student_status(sid)

        return generate_tool_answer(data)


    else:
        return "unable to answer"




In [4]:
# =====================
# RUN
# =====================
if __name__ == "__main__":

    vector_db = build_system()

    print("\nEdTech RAG Agent Ready\n")

    while True:

        q = input("Ask: ")

        if q == "exit":
            break

        print("\nAnswer:\n")

        print(edtech_assistant(q, vector_db))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



EdTech RAG Agent Ready


Answer:


Assignment must be submitted before deadline. Late submission allowed within 2 days with 10 percent penalty.

Sources:
['policy_assignment', 'faq_course', 'faq_certificate']


Answer:


Students can request refund within 7 days of enrollment. After 7 days refund is not allowed.

Sources:
['policy_refund', 'policy_refund', 'policy_enrollment']

